In [5]:
# 数据清洗，面经信息处理(原始面经处理，去掉html等脏数据)
import re
import pandas as pd
from cleantext import clean
import os
import jieba.posseg as pseg  # 用于分词和词性标注
from snownlp import SnowNLP

def clean_text(text):
    if isinstance(text, str):
        # 1) 去 HTML 标签
        text = re.sub(r'<[^>]+>', '', text)
        # 2) 去 URL & Email
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        # 3) 保留指定字符，删除其他字符
        keep_pat = re.compile(r'[^\u4e00-\u9fffA-Za-z0-9，。！？：、“”‘’（）【】《》_ . \\n]')
        text = keep_pat.sub('', text)
        # 4) 英文半角标点转中文全角
        trans = str.maketrans({
            ',': '，', '?': '？', '!': '！',
            ':': '：', ';': '；', '(': '（', ')': '）'
        })
        text = text.translate(trans)
        # 5) 清理多余空格
        text = clean(text, clean_all=False, extra_spaces=True)
    return text

# def judge_sentence(text: str) -> list[str]:
#     if not isinstance(text, str):
#         return []
#     # 1. 去掉数字序号
#     text = re.sub(r'\d+、', '', text)
#     # 2. 把换行强制变成句号，让 SnowNLP 一定切开
#     text = text.replace('\\n', '。')
#     return [s.strip() for s in SnowNLP(text).sentences if s.strip()]

def judge_sentence(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    # 统一分隔符：数字顿号、换行、中英文句末标点
    discourse = r'然后|再|接下来|并且|同时|另外|而且|再者|接着|先|包括|以及|但是|要是'
    pat = rf'{discourse}|\d+、|\d+ |[。！？；.!?]|\\n'
    return [s.strip() for s in re.split(pat, text) if s.strip()]


# 清洗汇总数据，去掉明显不合理的问题
# 1. 谓语识别函数（判断句子完整性核心）
def has_predicate(text):
    if not isinstance(text, str) or text.strip() == '':
        return False
    
    # 分词并获取词性（过滤标点和空格）
    words = pseg.cut(text.strip())
    pos_list = [pos for word, pos in words if word.strip() and pos not in ['x', 'w']]  # x:标点，w:空格
    
    # 谓语词性：动词v、形容词a、能愿动词vaux、判断动词vshi等
    predicate_pos = {'v', 'a', 'vaux', 'vshi'}
    
    # 检查是否包含任一谓语词性
    return len(set(pos_list) & predicate_pos) > 0

# 2. 完整句子判断函数（基于谓语）
def is_complete_sentence(text):
    text = str(text).strip()
    if not text:
        return False
    
    # 规则1：包含谓语成分（核心判断）
    if has_predicate(text):
        return True
    
    # 规则2：虽无谓语，但有句末标点且语义完整（特殊情况）
    if re.search(r'[。！？；]$', text) and len(text) > 5:
        return True
    
    return False

# 3. 新增过滤规则
def should_filter(text):
    text = str(text).strip()
    if not text:
        return True
    
    # 规则1：包含指定关键词
    filter_keywords = ['答：', 'sql题', '代码题', '算法题','算法']
    if any(keyword in text for keyword in filter_keywords):
        return True
    
    # 规则2：句子长度≤2个字
    if len(text) <= 4:
        return True
    
    # 规则3：关键词
    filter_keywords = ['介绍项目', '面试介绍','岗位面试时间']
    if any(keyword == text for keyword in filter_keywords):
        return True
    return False

def stand_text(text):
    text = re.sub(r'^\d+\.', '', str(text)).strip()
    return text


# 获取文件夹下的Excel文件
file_list = os.listdir('./面经')
# 初始化空DataFrame用于汇总（只保留clean_content列）
df_final = pd.DataFrame(columns=['clean_content'])

for f in file_list:
    # 只处理Excel文件，且排除已清洗的文件
    if f.endswith('xlsx') and not f.endswith('_clean.xlsx') and not f.endswith('汇总.xlsx') and not f.endswith('答案.xlsx'):
        file_path = os.path.join('./面经', f)
        df = pd.read_excel(file_path)
        
        # 清洗content列，生成clean_content
        df['clean_content'] = df['content'].apply(clean_text)
        
        # 提取英文单词（可选）
        pattern = re.compile(r'\b[A-Za-z]{2,}(?:-[A-Za-z]+)*\b')
        df['en_word'] = df['clean_content'].apply(
            lambda row: ','.join(set(pattern.findall(row))) if isinstance(row, str) else ''
        )
        
        # 筛选workTime小于2025的记录
        df = df[(df['workTime'] < 2025) & (df['workTime'] > 2020)]
        
        # 根据文件名筛选含特定关键词的记录
        if '数据' in f:
            keyword = '数据'
            mask = (
                df['clean_content'].str.contains(keyword, case=False, na=False) &
                df['title'].str.contains(keyword, case=False, na=False) &
                ~df['title'].str.contains('实习', case=False, na=False)
            )
            df = df[mask]
        elif '大模型' in f:
            keyword = '大模型'
            mask = (
                df['clean_content'].str.contains(keyword, case=False, na=False) &
                df['title'].str.contains(keyword, case=False, na=False) &
                ~df['title'].str.contains('实习', case=False, na=False)
            )
            df = df[mask]
        df.to_excel(f'./面经/{f[:-5]}_clean.xlsx')
        df = df[['clean_content']]
        df['clean_content'] = df['clean_content'].apply(judge_sentence)
        df = df.explode('clean_content').reset_index(drop=True)
        # print(df.head(10))
        # 使用concat按行拼接
        df_final = pd.concat([df_final, df[['clean_content']]], ignore_index=True)
        print(f"已处理 {f}，当前汇总数据量：{len(df_final)} 条")

# 4. 应用所有规则
df_final['has_predicate'] = df_final['clean_content'].apply(has_predicate)
df_final['is_complete'] = df_final['clean_content'].apply(is_complete_sentence)
df_final['to_filter'] = df_final['clean_content'].apply(should_filter)
df_final['clean_content'] = df_final['clean_content'].apply(stand_text)
df_final = df_final.drop_duplicates('clean_content')

# 5. 筛选结果：完整句子且不需要过滤
df_result = df_final[(df_final['is_complete']) & (~df_final['to_filter'])]

# 6. 输出结果
print("处理结果：")
print(df_result[['clean_content', 'has_predicate', 'is_complete', 'to_filter']])
print("\n最终保留的有效句子：")
print(df_result['clean_content'].tolist())
df_result.to_excel('./面经/汇总.xlsx')

已处理 面经.xlsx，当前汇总数据量：1236 条
处理结果：
                                          clean_content  has_predicate  \
0                                           之前面试了京东问题如下           True   
3                                                2问的问题，           True   
4                       介绍一下你的技术栈，说一下实时和离线开发的步骤，解决了哪些难点           True   
5     flink对于实时join如果两个表都很大如果做到实时更新，lookup join该怎么实现...           True   
6                     对于实时来说可以监控哪些任务，哪些地方可以设置监控，如何判断有延迟           True   
...                                                 ...            ...   
1226                               主要问实习项目和一些简单的八股，一题手撕           True   
1228            主要问业务理解和个人规划，也非常详细的拷打了实习项目和对某一个新业务开发的思路           True   
1229                                  这个HR面感觉还拷打了挺多技术细节           True   
1231                                拷打了实习项目，一点八股，手撕了一题，           True   
1232            根据手撕的题目进行了一些优化扩展，最后还问了一下个人对现在AI前景和岗位的理解           True   

      is_complete  to_filter  
0            True      False  
3            Tru

In [ ]:
# 清洗面经及答案，只保留正确答案，其余删掉

# 读取面经数据
import pandas as pd

data = pd.read_excel('./面经/牛客网数据面经&答案.xlsx')
for index, row in data.iterrows():
    key = row['key']
    title = row['title']
    answers = row['answer']
    correct_answers = []
    # 要 关键字正确答案之后，解答思路之前的句子
    answer_parts = answers.split('解答思路')
    print(key,title)
    if len(answer_parts) > 0:
        answer_section = answer_parts[0]
        answer_section = ' '.join(answer_section.splitlines())
        # 去掉“正确答案：”等前缀,只保留答案内容      
        answer_section = re.sub(r'^(正确答案[:：]\s*)', '', answer_section).strip()
        answer_section = answer_section.replace('- 正确答案：','').replace('-','')
        # 去掉开头的空格
        answer_section = answer_section.lstrip()
    # 加到data中
    data.at[index, 'answer'] = answer_section
# 保存清洗后的数据  
data.to_excel('./面经/牛客网数据面经&答案_cleaned.xlsx', index=False)

In [ ]:
"""
语义去重 + 快速相似度分布分析
优化点：用FAISS批量计算相似度，替代嵌套循环，速度提升10-100倍
依赖：pip install sentence-transformers scikit-learn faiss-cpu tqdm matplotlib
"""

import json, random
import jieba
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
import faiss
import pandas as pd
import hashlib
# 设置中文字体
plt.rcParams["font.family"] = ["SimHei", "WenQuanYi Micro Hei", "Heiti TC"]
plt.rcParams["axes.unicode_minus"] = False

# ========= 1. 读数据 =========
df = pd.read_excel('面经/牛客网数据面经&答案_cleaned.xlsx')
texts = df['title'].tolist()
texts = np.array(texts)  # <-- 新增
print(f"数据量：{len(texts)} 条")

# 抽样计算相似度（控制计算量，最多1000条）
sample_indices = list(range(len(texts)))
sample_texts = texts

# ========= 2. 文本编码 =========
model = SentenceTransformer("model/sungw111/text2vec-base-chinese-sentence")
print("编码文本中...")
emb = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb = emb.astype("float32")
sample_emb = emb[sample_indices]  # 抽样的向量

# ========= 3. 快速计算原始样本相似度分布（核心优化） =========
print("计算原始样本相似度分布（FAISS加速）...")
def fast_calculate_similarities(vectors, top_k=50):
    """用FAISS批量计算每个向量的Top-K相似向量，提取相似度值"""
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)  # 内积=余弦相似度（向量已归一化）
    index.add(vectors)
    
    # 每个向量找top_k个最相似的（包括自己）
    similarities, indices = index.search(vectors, top_k)
    
    # 提取相似度（排除自身，且只保留上三角部分避免重复）
    sim_values = []
    for i in range(len(vectors)):
        # 遍历当前向量的相似向量，只保留i < j的对（避免重复）
        for j, sim in zip(indices[i], similarities[i]):
            if j > i:  # 只记录j > i的对，避免重复计算和自身比较
                sim_values.append(sim)
    return sim_values

# 原始样本相似度（每个向量找前50个最相似的，足够反映分布）
original_similarities = fast_calculate_similarities(sample_emb, top_k=50)

# ========= 4. KMeans分簇 + 去重 =========

# keep_idx = []
# index = faiss.IndexFlatIP(emb.shape[1])
# threshold = 0.9
# print(f"簇内去重（阈值：{threshold}）...")
# for c in tqdm(range(n_clusters), desc="去重进度"):
#     mask = labels == c
#     if mask.sum() <= 1:
#         keep_idx.extend(np.where(mask)[0])
#         continue
#     cluster_emb = emb[mask]
#     index.reset()
#     index.add(cluster_emb)
#     D, I = index.search(cluster_emb, k=2)
#     skip = set()
#     for i, (dists, nbrs) in enumerate(zip(D, I)):
#         if i in skip:
#             continue
#         for j, sim in zip(nbrs[1:], dists[1:]):
#             if sim >= threshold:
#                 skip.add(j)
#     remain = [i not in skip for i in range(len(cluster_emb))]
#     keep_idx.extend(np.where(mask)[0][remain])

# ========= 4. 簇内两级去重：SimHash 粗排 + 向量精排 =========
def simhash_hash(text):
    """64-bit SimHash"""
    v = [0]*64
    for tok in jieba.lcut(text):
        h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
        for i in range(64):
            v[i] += 1 if (h>>i)&1 else -1
    return sum(1<<i for i in range(64) if v[i]>0)

def hamming_dist(a, b):
    return bin(a^b).count('1')

n_clusters = min(100, len(texts)//10)
print(f"聚类簇数：{n_clusters}")
kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1024, random_state=42)
labels = kmeans.fit_predict(emb)

keep_idx = []
index = faiss.IndexFlatIP(emb.shape[1])   # 用于第二步向量精排
threshold_vec = 0.90
threshold_ham = 3
print(f"SimHash Hamming≤{threshold_ham} + 向量余弦≥{threshold_vec} 两级去重...")

for c in tqdm(range(n_clusters), desc="去重进度"):
    mask = labels == c
    if mask.sum() <= 1:
        keep_idx.extend(np.where(mask)[0])
        continue

    cluster_idx = np.where(mask)[0]          # 本簇原始行号
    cluster_txt = texts[mask].tolist()

    # ---- ① SimHash 粗排 ----
    hashes = [simhash_hash(t) for t in cluster_txt]
    sim_skip = set()                         # 被 SimHash 判重
    for i in range(len(hashes)):
        if i in sim_skip:
            continue
        for j in range(i+1, len(hashes)):
            if hamming_dist(hashes[i], hashes[j]) <= threshold_ham:
                sim_skip.add(j)

    remain_after_sim = [i for i in range(len(cluster_idx)) if i not in sim_skip]
    if not remain_after_sim:                 # 全重，只留一条
        remain_after_sim = [0]

    # ---- ② 向量精排 ----
    coarse_idx = cluster_idx[remain_after_sim]   # 行号
    coarse_emb = emb[coarse_idx]
    index.reset()
    index.add(coarse_emb)

    D, I = index.search(coarse_emb, k=2)         # 找最像的 1 条（排除自己）
    vec_skip = set()
    for i, (dists, nbrs) in enumerate(zip(D, I)):
        if i in vec_skip:
            continue
        for j, sim in zip(nbrs[1:], dists[1:]):
            if sim >= threshold_vec:
                vec_skip.add(j)

    final_remain = [i not in vec_skip for i in range(len(coarse_idx))]
    keep_idx.extend(coarse_idx[final_remain])

# ========= 5. 快速计算去重后相似度分布 =========
print("计算去重后相似度分布...")
dedup_emb = emb[keep_idx]
dedup_sample_emb = dedup_emb
# 去重后相似度（同样用FAISS加速）
dedup_similarities = fast_calculate_similarities(dedup_sample_emb, top_k=50)

# ========= 6. 可视化对比 =========
plt.figure(figsize=(12, 6))
plt.hist(original_similarities, bins=50, alpha=0.5, label=f'原始样本 (n={len(texts)})', density=True)
plt.hist(dedup_similarities, bins=50, alpha=0.5, label=f'去重后 (n={len(keep_idx)})', density=True)
plt.axvline(x=threshold_vec, color='r', linestyle='--', label=f'阈值 {threshold_vec}')
plt.xlabel('语义相似度')
plt.ylabel('概率密度')
plt.title('相似度分布对比（FAISS加速版）')
plt.xlim(0.5, 1.0)
plt.legend()
plt.savefig('similarity_distribution_fast.png', dpi=300)
plt.show()

# ========= 7. 保存结果 =========
df_clean = df.iloc[keep_idx]          # 按行号抽取
out_xlsx = "面经/牛客网数据面经&答案_dedup.xlsx"
df_clean.to_excel(out_xlsx, index=False)
print(f"去重后数据已保存 -> {out_xlsx}  共 {len(df_clean)} 条")
# 输出统计
print(f"\n原始数量：{len(texts)} 条")
print(f"去重后：{len(keep_idx)} 条（减少 {len(texts)-len(keep_idx)} 条）")
print(f"原始高相似度比例（≥{threshold_vec}）：{sum(s >= threshold_vec for s in original_similarities)/len(original_similarities):.2%}")
print(f"去重后高相似度比例：{sum(s >= threshold_vec for s in dedup_similarities)/len(dedup_similarities):.2%}") 

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
7-Dimension Data Quality Score
usage: python dq_score.py  # 默认读 raw_corpus.csv，输出 clean_corpus.csv
"""

import re, hashlib, math, json, tqdm, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ================== 1. 参数 ==================
MAX_LEN      = 2048
MIN_LEN      = 5
NON_ZH_RATIO = 0.25          # 允许最大非中文字符比
AD_KEYWORDS  = {'加微信', '红包', '内推', '扫码', '优惠券', '加我', 'q群'}
KEYWORDS = {
    # ----- 大模型 LLM -----
    'llm', 'gpt', 'bert', 'transformer', 'attention', 'self_attention',
    'lora', 'qlora', 'peft', 'fine_tune', 'sft', 'rlhf', 'instruct',
    'token', 'embedding', 'positional_encoding', 'ffn', 'layer_norm',
    'dropout', 'softmax', 'gelu', 'relu', 'sigmoid', 'adamw', 'adam',
    'scheduler', 'warmup', 'batch_size', 'gradient_accumulation',
    'mixed_precision', 'fp16', 'bf16', 'tensor_parallel', 'pipeline_parallel',
    'data_parallel', 'deepspeed', 'zero', 'offload', 'checkpoint',
    'kv_cache', 'flash_attention', 'rope', 'alibi', 'gpt3', 'gpt4',
    'chatglm', 'baichuan', 'llama', 'alpaca', 'vicuna', 'falcon',
    'prompt', 'prompt_template', 'few_shot', 'zero_shot', 'chain_of_thought',
    'rag', 'retriever', 'index', 'faiss', 'milvus', 'embedding_model',
    'cosine_similarity', 'rerank', 'cross_encoder', 'generation',
    'temperature', 'top_p', 'top_k', 'beam_search', 'do_sample',
    'hallucination', 'perplexity', 'bleu', 'rouge', 'distinct',
    'reward_model', 'critic', 'actor', 'ppo', 'dpo', 'kto',

    # ----- 数据仓库 & 大数据 -----
    'data_warehouse', 'dwd', 'dws', 'ads', 'dim', 'ods', 'etl', 'elt',
    'sql', 'hive', 'spark', 'spark_sql', 'pyspark', 'flink', 'kafka',
    'hadoop', 'hdfs', 'yarn', 'mapreduce', 'tez', 'impala', 'presto',
    'clickhouse', 'doris', 'starrocks', 'hbase', 'redis', 'mongodb',
    'mysql', 'postgresql', 'oracle', 'sqlserver', 'tidb', 'oceanbase',
    'data_lake', 'lakehouse', 'iceberg', 'hudi', 'delta_lake', 'paimon',
    'binlog', 'cdc', 'canal', 'debezium', 'airflow', 'dolphinscheduler',
    'azkaban', 'oozie', 'crontab', 'shell', 'git', 'gitlab', 'github',
    'parquet', 'orc', 'avro', 'json', 'csv', 'tsv', 'xml', 'protobuf',
    'partition', 'bucket', 'shuffle', 'broadcast', 'sort_merge',
    'cube', 'rollup', 'grouping_sets', 'window_function', 'rank', 'row_number',
    'lag', 'lead', 'udf', 'udaf', 'udtf', 'hive_function', 'spark_function',
    'slow_query', 'explain', 'index', 'btree', 'hash_index', 'columnar',
    'star_schema', 'snowflake_schema', 'fact_table', 'dimension_table',
    'surrogate_key', 'natural_key', 'scd', 'scd_type2', 'cdc_merge',
    'data_quality', 'data_profiling', 'anomaly_detection', 'completeness',
    'consistency', 'timeliness', 'uniqueness', 'validity', 'accuracy',
    'lineage', 'impact_analysis', 'data_governance', 'metadata',
    'atlas', 'datahub', 'metacat', 'griffin', 'great_expectations',
    'dbt', 'sqlmesh', 'looker', 'tableau', 'superset', 'quick_bi',
    'aliyun_maxcompute', 'aws_redshift', 'gcp_bigquery', 'azure_synapse',
    'starrocks_connector', 'flink_connector', 'kafka_connector',
    '大模型', '大语言模型', '预训练', '微调', '指令微调', '人类反馈', '强化学习', 'LoRA', 'QLoRA',
    '提示词', '提示模板', '少样本', '零样本', '思维链', '上下文', 'token', '嵌入', '词向量',
    '位置编码', '注意力', '自注意力', '多头注意力', '前馈网络', '层归一化', ' dropout',
    'softmax', '激活函数', 'AdamW', '学习率', '预热', '梯度累积', '混合精度', 'FP16', 'BF16',
    '张量并行', '流水线并行', '数据并行', 'DeepSpeed', 'ZeRO', '检查点', 'KV缓存', 'FlashAttention',
    'RoPE', 'ALiBi', 'GPT', 'ChatGLM', 'Baichuan', 'LLaMA', 'Alpaca', 'Vicuna',
    '温度采样', 'Top_P', 'Top_K', '束搜索', '幻觉', '困惑度', 'BLEU', 'ROUGE',
    '奖励模型', 'PPO', 'DPO', 'RAG', '检索器', '向量库', 'Faiss', 'Milvus', '重排',
    '数据仓库', '数仓', 'ODS', 'DWD', 'DWS', 'ADS', '维度表', '事实表', '缓慢变化维',
    'ETL', 'ELT', '数据集成', '数据清洗', '数据建模', '星型模型', '雪花模型',
    'Hive', 'Spark', 'Flink', 'Kafka', 'Hadoop', 'HDFS', 'YARN', 'MapReduce',
    'ClickHouse', 'Doris', 'StarRocks', 'HBase', 'Redis', 'MySQL', 'PostgreSQL',
    '数据湖', '湖仓一体', 'Iceberg', 'Hudi', 'DeltaLake', 'Paimon', 'Binlog', 'CDC',
    'Airflow', '海豚调度', '数据质量', '数据治理', '元数据', '数据血缘', '主键',
    '分区', '分桶', '索引', '布隆过滤器', '列式存储', 'Parquet', 'ORC', 'Avro',
    '窗口函数', '排名', '累计和', 'UDF', 'UDAF', 'UDTF', 'Explain', '执行计划',
    '慢查询', '数据倾斜', 'Shuffle', '广播', '排序合并', 'Cube', 'Rollup',
    '一致性', '完整性', '唯一性', '及时性', '有效性', '异常检测', '数据探查',
    'DBT', 'SQLMesh', 'Tableau', 'QuickBI', 'MaxCompute', 'BigQuery', 'Redshift'
}   # 可自行扩充
WEIGHT = {'len':0.10, 'encode':0.10, 'ad':0.15, 'dup':0.30,
          'sem':0.15, 'kw':0.1, 'fmt':0.10}  # 权重和=1
THRESHOLD = 0.75            # 高质量线

# ================== 2. 工具函数 ==================
def simhash_hash(text):
    v = [0]*64
    for tok in text.split():
        h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
        for i in range(64):
            v[i] += 1 if (h >> i) & 1 else -1
    return sum(1 << i for i in range(64) if v[i] > 0)

def hamming_dist(h1, h2):
    return bin(int(h1) ^ int(h2)).count('1')

# ------------------ 各维度打分 ------------------
# 长度得分：在合理范围内得1分，否则0分
def len_score(texts):
    return texts.apply(lambda x: 1 if MIN_LEN <= len(x) <= MAX_LEN else 0)

# 编码得分：非中文字符比例过高扣分
def encode_score(texts):
    def _enc(x):
        zh = len(re.findall(r'[\u4e00-\u9fff]', x))
        ratio = (len(x) - zh) / len(x) if len(x) else 1
        return max(0, 1 - ratio / NON_ZH_RATIO) if ratio < NON_ZH_RATIO else 0
    return texts.apply(_enc)

# 广告得分：含广告关键词扣分
def ad_score(texts):
    return texts.apply(lambda x: 0 if any(k in x for k in AD_KEYWORDS) else 1)

# 重复得分：近窗口内有相似文本扣分
def dup_score(texts, win=100, ham=3):
    hashes = texts.apply(simhash_hash)
    scores = []
    for i, h in enumerate(hashes):
        dup = any(hamming_dist(h, hashes[j]) < ham for j in range(max(0, i-win), i))
        scores.append(0 if dup else 1)
    return pd.Series(scores, index=texts.index)

# 语义得分：与均值向量余弦相似度
def sem_score(texts, batch=256):
    model = SentenceTransformer("model/sungw111/text2vec-base-chinese-sentence")
    print("编码文本中...")
    embs = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    # embs = MODEL.encode(texts.tolist(), batch_size=batch, normalize_embeddings=True, show_progress_bar=False)
    center = embs.mean(axis=0, keepdims=True)
    cos = cosine_similarity(embs, center).squeeze()
    return (cos - cos.min()) / (cos.max() - cos.min() + 1e-8)

# 关键词得分：TF-IDF 前10关键词得分和
def kw_score(texts):
    vectorizer = TfidfVectorizer(vocabulary=KEYWORDS, token_pattern=r'(?u)\b\w+\b')
    tfidf = vectorizer.fit_transform(texts)
    scores = [np.sort(row.data)[-10:].sum() for row in tfidf]
    scores = np.array(scores)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

# 格式得分
def fmt_score(texts):
    # 简易格式：必须含问号且 15<=len<=200
    return texts.apply(lambda x: 1 if ('?' in x or '？' in x) and 15<=len(x)<=200 else 0)

# ================== 3. 主函数 ==================
def dq_score(df, text_col='title'):
    df = df.copy()
    texts = df[text_col].astype(str)

    ls = len_score(texts)
    es = encode_score(texts)
    ad = ad_score(texts)
    dp = dup_score(texts)
    sm = sem_score(texts)
    kw = kw_score(texts)
    ft = fmt_score(texts)

    df['dq_score'] = (WEIGHT['len']*ls + WEIGHT['encode']*es +
                      WEIGHT['ad']*ad + WEIGHT['dup']*dp +
                      WEIGHT['sem']*sm + WEIGHT['kw']*kw + WEIGHT['fmt']*ft
                      )
    return df

# ================== 4. IO & 可视化 ==================
def main():
    in_file  = '面经\牛客网数据面经&答案_dedup.xlsx'   # 需含列 text
    out_file = '面经\牛客网数据面经&答案_double_cleaned.xlsx'

    df = pd.read_excel(in_file)
    print('>>> 原始数据:', len(df))
    df = dq_score(df)
    good = df[df.dq_score >= THRESHOLD].copy()
    print('>>> 高质量数据:', len(good), f'保留率 {len(good)/len(df):.1%}')
    good.to_excel(out_file, index=False)

    # 画分布
    plt.hist(df['dq_score'], bins=50, alpha=0.7, label='all')
    plt.hist(good['dq_score'], bins=50, alpha=0.7, label='clean')
    plt.axvline(THRESHOLD, color='red', linestyle='--')
    plt.legend(); plt.xlabel('DQ Score'); plt.ylabel('Count')
    plt.title('Data Quality Distribution')
    plt.savefig('dq_dist.png', dpi=200); plt.close()
    print('>>> 分布图已保存：dq_dist.png')

if __name__ == '__main__':
    main()

>>> 原始数据: 5671
编码文本中...


Batches: 100%|██████████| 89/89 [00:55<00:00,  1.61it/s]
d:\ProgramFiles\python\Lib\site-packages\sklearn\feature_extraction\text.py:1368: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


>>> 高质量数据: 1971 保留率 34.8%
>>> 分布图已保存：dq_dist.png


In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SFT 语料拆分 - 零过滤版
python sft_split.py  面经.xlsx  sft_corpus.jsonl
"""
import re, json, argparse
import pandas as pd

def main():
    df = pd.read_excel('面经\牛客网数据面经&答案_double_cleaned.xlsx')
    # 如果列名不是 Q/A 就改下面两行
    qa_pairs = df[['title', 'answer']].dropna()       # 去掉空行
    sft = [{'instruction': str(q).strip(),
            'input': '',
            'output': str(a).strip()} for q, a in qa_pairs.itertuples(index=False)]

    with open('面经\sft_corpus.jsonl', 'w', encoding='utf-8') as f:
        for item in sft:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f'✅ 转换完成！共 {len(sft)} 条指令对 → 面经\sft_corpus.jsonl')
if __name__ == '__main__':
    main()

✅ 转换完成！共 1971 条指令对 → 面经\sft_corpus.jsonl


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
1. 从文本提取 JSON 块 → 抽 QA
2. 读取 Excel 指定两列追加 QA
3. 过滤垃圾回答
4. 输出 qa_knowledge.jsonl
"""
import re
import json
import pathlib
import sys
import openpyxl

# ========== 参数 ==========
in_txt   = pathlib.Path("answer.jsonl")          # 原始文本
in_excel = pathlib.Path("面经\面经及其答案\保留的面经.xlsx")          # Excel 文件
out_file = pathlib.Path("qa_knowledge.jsonl")

sheet_name = 0       # 工作表索引或名称
q_col      = 3       # question 列号（1起始）
a_col      = 4       # answer   列号
skip_head  = True    # 是否跳过表头

# 垃圾回答关键词
FILTER_PAT = re.compile(r'无需回答|无明确问题|供他人参考|记忆模糊|无法回忆|省略一些问题', re.I)

# ========== 工具函数 ==========
# 匹配 JSON 对象或数组（支持嵌套，无需递归正则）
json_regex = re.compile(r'(\{(?:[^{}]|\{[^{}]*\})*\}|\[(?:[^\[\]]|\[[^\[\]]*\])*\])', re.S)

def parse_json_snippets(text: str):
    for m in json_regex.finditer(text):
        try:
            yield json.loads(m.group())
        except Exception:
            continue

def iter_qa(obj):
    """递归抽 question/answer 或 问题/回答/简答/description"""
    if isinstance(obj, dict):
        q = obj.get("question") or obj.get("问题") or obj.get("clean_content")
        a = obj.get("answer") or obj.get("回答") or obj.get("description") or obj.get("简答")
        if isinstance(q, str) and isinstance(a, str):
            q, a = q.strip(), a.strip()
            if q and a and not FILTER_PAT.search(a):
                yield {"question": q, "answer": a}
        for v in obj.values():
            yield from iter_qa(v)
    elif isinstance(obj, list):
        for item in obj:
            yield from iter_qa(item)

def read_excel_qa():
    """读取 Excel 指定两列 → 生成 QA dict"""
    if not in_excel.exists():
        return
    wb = openpyxl.load_workbook(in_excel, data_only=True)
    if isinstance(sheet_name, int):
        ws = wb.worksheets[sheet_name]
    else:
        ws = wb[sheet_name]

    rows = ws.iter_rows(min_row=2 if skip_head else 1, values_only=True)
    for row in rows:
        if len(row) < max(q_col, a_col):
            continue
        q, a = row[q_col - 1], row[a_col - 1]
        if isinstance(q, str) and isinstance(a, str):
            q, a = q.strip(), a.strip()
            if q and a and not FILTER_PAT.search(a):
                yield {"question": q, "answer": a}

def main():
    all_qa, seen = [], set()

    # 1. 从文本 JSON 抽取
    if in_txt.exists():
        text = in_txt.read_text(encoding="utf-8")
        for j in parse_json_snippets(text):
            for qa in iter_qa(j):
                key = (qa["question"], qa["answer"])
                if key not in seen:
                    seen.add(key)
                    all_qa.append(qa)
    print(f"✅ 共抽取 {len(all_qa)} 条干净问答，已保存到 {out_file.resolve()}")
    # 2. 从 Excel 追加
    for qa in read_excel_qa():
        key = (qa["question"], qa["answer"])
        if key not in seen:
            seen.add(key)
            all_qa.append(qa)

    if not all_qa:
        print("⚠️  未抽到任何有效问答")
        return

    # 3. 写出 jsonl
    with out_file.open("w", encoding="utf-8") as f:
        for qa in all_qa:
            f.write(json.dumps(qa, ensure_ascii=False) + "\n")

    print(f"✅ 共抽取 {len(all_qa)} 条干净问答，已保存到 {out_file.resolve()}")

if __name__ == "__main__":
    main()

In [ ]:
# 调用模型生成问题，失败，原因：内存不够

import json, os, gc
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# ---------- 1. 单例模型 + 最省配置 ----------
model_dir = r".\langboat\mengzi-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_dir)
model = T5ForConditionalGeneration.from_pretrained(model_dir)
model.eval()                                 # 关 dropout
torch.set_grad_enabled(False)                # 不计算梯度
# 若内存 < 4 GB 可再开半精度
# model = model.half()

# ---------- 2. 生成一条就立刻写盘 ----------
def generate_and_stream(in_file, out_file):
    with open(out_file, "w", encoding="utf-8") as fw:
        for idx, line in enumerate(open(in_file, encoding="utf-8"), 1):
            rec = json.loads(line)
            answer = rec["answer"][:128]
            prompt = f"答案：{answer} 问题："
            inputs = tokenizer(prompt, return_tensors="pt")
            with torch.no_grad():
                out_ids = model.generate(**inputs, max_length=64, do_sample=False)
            question = tokenizer.decode(out_ids[0], skip_special_tokens=True)

            fw.write(json.dumps({"question": question, "answer": rec["answer"]}, ensure_ascii=False) + "\n")
            fw.flush()          # 立即落盘，不积压内存
            if idx % 100 == 0:
                print(f"{idx} 条完成")
                gc.collect()    # 强制回收 Python 对象 & PyTorch 缓存

# ---------- 3. 启动 ----------
generate_and_stream("qa_knowledge.jsonl", "auto_questions.jsonl")
print("✅ 全部生成完毕，文件已关闭。")

In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
1. 用正则从 kb.jsonl 生成 400 题
2. 直接写 Excel gold_400.xlsx
3. 你只管标 label 列
"""
import json
import random
import re
import pandas as pd

# ---------- 参数 ----------
kb_file = "qa_knowledge.jsonl"
excel_file = "gold_400.xlsx"
random.seed(62)          # 可复现

# ---------- 1. 正则模板 ----------
templates = [
    "什么是{}？","{}的定义是？","关于{}，请说明。","{}包括哪些内容？",
    "{}的流程是怎样的？","{}有哪些注意事项？","{}和什么有关？",
    "{}适用于哪些场景？","{}的优势是什么？","{}的缺点有哪些？"
]

def extract_keyword(text: str, max_len: int = 40) -> str:
    txt = text[:max_len]
    return re.split(r"[，。；！？\s]", txt, 1)[0].strip() or txt.strip()

def make_question(answer: str) -> str:
    # 1. 取前 60 字，按标点切第一句
    sent = re.split(r"[。！？；]", answer[:60])[0]
    # 2. 若太短，再往后凑
    if len(sent) < 10:
        sent = answer[:60]
    # 3. 直接当问题
    return sent + "？"

# ---------- 2. 逐行读取并生成 ----------
all_qa = []
for line in open(kb_file, encoding="utf-8"):
    rec = json.loads(line)
    answer = rec["answer"] if ' ' not in rec["answer"] else rec["answer"].split(' ')[1]
    # 取最后一个冒号之的全部内容，没有冒号则原样返回
    # 先尝试取「答案：」之后；没有就取第一个冒号之后；再没有就原样
    full = rec["answer"]
    if '答案：' in answer:
        answer = answer.split('答案：')[-1]
    elif '：' in answer:
        answer = answer.split('：', 1)[1]   # 只切一次，取后半
    else:
        answer = answer
    question = make_question(answer)
    if len(question) > 20:
        all_qa.append({"question": question, "answer": answer})

# ---------- 3. 随机抽 400 条 ----------
sample_400 = random.sample(all_qa, 400)

# ---------- 4. 写 Excel ----------
df = pd.DataFrame(sample_400)
df["label"] = ""          # 待标注列
df.to_excel(excel_file, index=False, engine="openpyxl")

print("✅ 已生成并写入 → ", excel_file)
print("   请在 Excel 里给「label」列打 1（正确）或 0（错误）即可！")
# out_file = "gold_400.xlsx"
# df1 = pd.read_excel('gold_4001.xlsx')
# df2 = pd.read_excel('gold_4002.xlsx')
# random.seed(42)        # 可复现
# total = pd.concat([df1, df2], ignore_index=True)

# # 强制转数字，空值视为0
# total["label"] = pd.to_numeric(total["label"], errors="coerce").fillna(0).astype(int)

# # ========== 3. 保证正例≥75 ==========
# pos = total[total["label"] == 1]
# neg = total[total["label"] == 0]

# if len(pos) < 75:
#     raise ValueError(f"正例只有{len(pos)}条，不足75条，请再多标一些！")

# # 按最小比例抽
# need_pos = 75
# need_neg = 400 - need_pos

# # 若负例不够，全拿负例，剩余用正例补
# if len(neg) < need_neg:
#     need_neg = len(neg)
#     need_pos = 400 - need_neg

# sample_pos = pos.sample(n=need_pos, random_state=42)
# sample_neg = neg.sample(n=need_neg, random_state=42)
# gold_400 = pd.concat([sample_pos, sample_neg]).sample(frac=1, random_state=42).reset_index(drop=True)

# # ========== 4. 写 Excel ==========
# gold_400.to_excel(out_file, index=False, engine="openpyxl")
# print("✅ 已写入 → ", out_file)
# print(f"   正例：{need_pos} 条，负例：{need_neg} 条，总计 400 条。")


✅ 已生成并写入 →  gold_400.xlsx
   请在 Excel 里给「label」列打 1（正确）或 0（错误）即可！


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
1. 读取 gold_400.xlsx（已标注）
2. 算 Precision@1 & Recall 估算
3. 输出错例清单 excel
"""
import pandas as pd
import json
import numpy as np
from sklearn.metrics import precision_score
from transformers import AutoTokenizer, AutoModel
import torch
import tqdm

# ---------- 1. 读标注 ----------
gold = pd.read_excel("gold_400.xlsx")
gold["label"] = gold["label"].astype(int)

# ---------- 2. 句向量相似度（纯 CPU） ----------
model_dir = "sungw111/text2vec-base-chinese-sentence"  # 提前下好 bert-base-chinese
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModel.from_pretrained(model_dir)
model.eval()
torch.set_grad_enabled(False)

def embed(text: str):
    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True)
    with torch.no_grad():
        vec = model(**inputs).last_hidden_state.mean(dim=1)
    return vec.squeeze().numpy()

# 预嵌入知识库（一次性，可缓存）
kb = [json.loads(l) for l in open("qa_knowledge.jsonl", encoding="utf-8")]
kb_text = [rec["answer"][:128] for rec in kb]
kb_vec = np.array([embed(t) for t in tqdm.tqdm(kb_text, desc="Embedding KB")])

# ---------- 3. 逐条算召回 & 正确性 ----------
y_true, y_pred = [], []
recall_flags = []

for _, row in tqdm.tqdm(gold.iterrows(), total=400, desc="Eval"):
    q, a, label = row["question"], row["answer"], row["label"]
    # 召回：库里是否有相似 chunk
    q_vec = embed(q)
    sims = np.dot(kb_vec, q_vec) / (np.linalg.norm(kb_vec, axis=1) * np.linalg.norm(q_vec))
    recall_flags.append(1 if sims.max() > 0.65 else 0)

    # 正确性：用规则「关键句包含」近似
    key_sent = a[:30]                     # 取前 30 字当关键句
    pred_label = 1 if key_sent in a else 0
    y_true.append(label)
    y_pred.append(pred_label)

# ---------- 4. 指标 ----------
prec = precision_score(y_true, y_pred)      
recall_est = np.mean(recall_flags)

print("========== 评估结果 ==========")
print(f"Precision@1 : {prec:.3f}")
print(f"Recall 估算 : {recall_est:.3f}")

# ---------- 5. 错例清单 ----------
error_df = gold[gold["label"] != y_pred]
error_df.to_excel("error_cases.xlsx", index=False)
print("错例已导出 → error_cases.xlsx")

d:\ProgramFiles\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\ProgramFiles\python\Lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] 找不到指定的程序。'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Eval: 100%|██████████| 400/400 [00:23<00:00, 16.83it/s]


========== 评估结果 ==========
Precision@1 : 0.752
Recall 估算 : 1.000
错例已导出 → error_cases.xlsx
